In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import scipy.signal as signal
from tqdm import tqdm

# =========================
# 路径配置
# =========================
ROOT = r"D:\Project_Github\audio_click_mil"
AUDIO_DIR = os.path.join(ROOT, "data", "original_audio")
CSV_PATH = os.path.join(ROOT, "processed_data", "all_bags.csv")
SAVE_DIR = os.path.join(ROOT, "processed_data", "mfcc")

os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# 参数
# =========================
TARGET_SR = 48000
BAG_DURATION = 60       # 秒
INSTANCE_DURATION = 1   # 秒
N_MELS = 40             # 方案B
TARGET_SIZE = 128       # resize到128x128

# =========================
# 高通滤波器（5kHz）
# =========================
def highpass_filter(y, sr, cutoff=5000, order=6):
    sos = signal.butter(order, cutoff, btype='highpass', fs=sr, output='sos')
    return signal.sosfilt(sos, y)

# =========================
# log-mel提取 + resize
# =========================
def extract_logmel(y, sr):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=1024,
        hop_length=512,
        n_mels=N_MELS,
        fmin=5000,
        fmax=sr//2
    )

    logmel = librosa.power_to_db(mel)

    # resize到 (128,128)
    logmel_resized = librosa.util.fix_length(logmel, size=TARGET_SIZE, axis=1)
    logmel_resized = librosa.util.fix_length(logmel_resized, size=TARGET_SIZE, axis=0)

    return logmel_resized.astype(np.float32)

# =========================
# 主处理流程
# =========================
def process():
    df = pd.read_csv(CSV_PATH)

    grouped = df.groupby("audio_file")

    for audio_file, group in tqdm(grouped):
        audio_path = os.path.join(AUDIO_DIR, audio_file)

        if not os.path.exists(audio_path):
            print(f"Missing: {audio_path}")
            continue

        # ===== 读取音频 =====
        y, sr = librosa.load(audio_path, sr=None)

        # ===== 重采样 =====
        if sr != TARGET_SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
            sr = TARGET_SR

        # ===== 高通滤波 =====
        y = highpass_filter(y, sr)

        # ===== 输出目录 =====
        file_name = audio_file.replace(".wav", "")
        save_folder = os.path.join(SAVE_DIR, file_name)
        os.makedirs(save_folder, exist_ok=True)

        # ===== 遍历bag =====
        for _, row in group.iterrows():
            bag_idx = int(row["bag_idx"])
            start = int(row["bag_start_sec"] * sr)
            end = int(row["bag_end_sec"] * sr)

            bag_audio = y[start:end]

            instances = []

            for i in range(60):
                s = int(i * INSTANCE_DURATION * sr)
                e = int((i+1) * INSTANCE_DURATION * sr)

                inst = bag_audio[s:e]

                # 防止长度不足
                if len(inst) < sr:
                    inst = np.pad(inst, (0, sr - len(inst)))

                feat = extract_logmel(inst, sr)  # (128,128)
                instances.append(feat)

            bag_array = np.stack(instances, axis=0)  # (60,128,128)

            save_path = os.path.join(save_folder, f"bag_{bag_idx:03d}.npy")
            np.save(save_path, bag_array)

    print("✅ MFCC (log-mel) 生成完成")


if __name__ == "__main__":
    process()

  0%|          | 0/34 [00:00<?, ?it/s]d:\Python_env\Dolphin\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 34/34 [04:33<00:00,  8.05s/it]

✅ MFCC (log-mel) 生成完成
